In [1]:
from mava.networks.retention import MultiScaleRetention
from omegaconf import DictConfig
import jax
import jax.numpy as jnp
import copy

# jax.config.update("jax_enable_x64", True)

bsz = 16
num_agents = 4
obs_dim = 11
num_time_steps = 100
seq_len = num_agents * num_time_steps

retnet_embed_dim = 32
retnet_num_heads = 1

2025-02-27 12:26:59.737257: W external/xla/xla/service/gpu/nvptx_compiler.cc:765] The NVIDIA driver's CUDA version is 12.4 which is older than the ptxas CUDA version (12.8.61). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
/home/ruanjohn/miniconda3/envs/mava/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
memory_config = DictConfig(
    {
        "type": "rec_sable",
        "decay_scaling_factor": 0.3,
        "timestep_positional_encoding": True,
        "timestep_chunk_size": None,
    }
)

decay_kappas = 1 - jnp.exp(jnp.linspace(jnp.log(1 / 32), jnp.log(1 / 512), retnet_num_heads))
decay_kappas *= memory_config.decay_scaling_factor
decay_kappas = jnp.log(decay_kappas)
decay_kappas = decay_kappas[None, :, None, None]

In [3]:
msr = MultiScaleRetention(
    embed_dim=retnet_embed_dim,
    n_head=retnet_num_heads,
    n_agents=num_agents,
    memory_config=memory_config,
    masked=False,
    decay_scaling_factor=memory_config.decay_scaling_factor,
)

In [4]:
key = jax.random.PRNGKey(0)
key, subkey = jax.random.split(key)

obs = jax.random.normal(subkey, (bsz, seq_len, retnet_embed_dim))

# assuming no resets
dones = jnp.zeros((bsz, seq_len), dtype=bool)

init_hstate = jnp.zeros(
    (
        bsz,
        retnet_num_heads,
        retnet_embed_dim // retnet_num_heads,
        retnet_embed_dim // retnet_num_heads,
    )
)
step_counts = jnp.arange(num_time_steps)
step_counts = step_counts[None, ...].repeat(bsz, axis=0)[..., None].repeat(num_agents, axis=-1)
step_counts = step_counts.reshape(bsz, seq_len)

In [5]:
key, init_key = jax.random.split(key)
params = msr.init(
    init_key,
    obs,
    obs,
    obs,
    init_hstate,
    dones,
    step_counts,
)

In [6]:
hstate = copy.deepcopy(init_hstate)
act_output = []


# for the decoder we use the chunkwise
for step in range(num_time_steps):
    # todo: reset later
    hstate = hstate * jnp.exp(decay_kappas)
    obs_i = obs[:, step * num_agents : (step + 1) * num_agents, ...]
    dones_i = dones[:, step * num_agents : (step + 1) * num_agents]
    step_counts_i = step_counts[:, step * num_agents : (step + 1) * num_agents]

    out, hstate = msr.apply(params, obs_i, obs_i, obs_i, hstate, step_counts_i, method="recurrent")
    act_output.append(out)

In [7]:
act_output = jnp.concatenate(act_output, axis=1)

In [8]:
act_output.shape

(16, 400, 32)

In [9]:
hstate = copy.deepcopy(init_hstate)
train_out, _ = msr.apply(params, obs, obs, obs, hstate, dones, step_counts)

In [10]:
train_out.shape

(16, 400, 32)

In [11]:
total_error = jnp.mean(jnp.abs(train_out - act_output))
total_error

Array(4.4014446e-06, dtype=float32)

In [12]:
jnp.abs(train_out - act_output)

Array([[[2.54437327e-06, 3.23913991e-06, 3.69315967e-06, ...,
         6.68503344e-06, 7.26431608e-07, 2.85543501e-06],
        [1.04308128e-07, 1.46217644e-05, 6.06104732e-06, ...,
         1.99358910e-05, 3.24938446e-06, 1.69500709e-07],
        [1.62795186e-06, 2.12527812e-06, 1.03889033e-06, ...,
         5.45568764e-06, 9.68575478e-07, 3.96277755e-06],
        ...,
        [2.18022615e-06, 1.03376806e-07, 3.04030254e-06, ...,
         1.43796206e-06, 6.57746568e-06, 5.22937626e-06],
        [1.39046460e-06, 2.38418579e-06, 1.23493373e-06, ...,
         1.10641122e-06, 2.92132609e-06, 5.13903797e-06],
        [5.02914190e-08, 7.26431608e-08, 1.86264515e-09, ...,
         2.22586095e-07, 7.91624188e-08, 2.25380063e-07]],

       [[6.06290996e-07, 4.23518941e-06, 5.96977770e-06, ...,
         3.65264714e-06, 2.63843685e-06, 4.72180545e-06],
        [5.20376489e-07, 4.44985926e-06, 7.36210495e-06, ...,
         3.54833901e-07, 5.14555722e-06, 5.14322892e-06],
        [9.91858542e-07, 